# Training Data Evaluation Report
This notebook evaluates all trained models in the `training_data` folder against the images in the same folder structure. It generates a comprehensive pivot report comparing all models side-by-side for each image.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import xisf
import os
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import concurrent.futures
import multiprocessing

# --- Device Setup ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
TRAINING_DATA_DIR = Path("/Storage/Files/practicalML/gitlab/practicalml/training_data")
IMAGE_EXT = ".xisf"
BATCH_SIZE = 256  # Optimized for 5080
NUM_WORKERS = min(16, os.cpu_count()) # Parallel workers for batch collation if needed

# Transform for inference
transform = transforms.Compose([
    transforms.ToTensor(),
])

In [ ]:
def load_inference_model(model_path):
    """Loads a PyTorch model from a .pth file."""
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    arch_name = str(checkpoint.get('model_name', 'unknown_model')).lower()
    num_classes = checkpoint.get('num_classes', 7)
    input_channels = checkpoint.get('input_channels', 1)
    class_names = checkpoint.get('class_names', [])
    
    # Build Model Architecture
    if 'mobilenet' in arch_name:
        model = torchvision.models.mobilenet_v2(weights=None, num_classes=num_classes)
        if input_channels != 3:
            orig = model.features[0][0]
            model.features[0][0] = nn.Conv2d(input_channels, orig.out_channels, orig.kernel_size, orig.stride, orig.padding, bias=False)
    elif 'resnet' in arch_name:
        model = torchvision.models.resnet18(weights=None, num_classes=num_classes)
        if input_channels != 3:
            model.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    elif 'densenet' in arch_name:
        model = torchvision.models.densenet121(weights=None)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        if input_channels != 3:
            orig = model.features.conv0
            model.features.conv0 = nn.Conv2d(input_channels, orig.out_channels, orig.kernel_size, orig.stride, orig.padding, bias=False)
    elif 'shufflenet' in arch_name:
        model = torchvision.models.shufflenet_v2_x1_0(weights=None, num_classes=num_classes)
        if input_channels != 3:
            orig = model.conv1[0]
            model.conv1[0] = nn.Conv2d(input_channels, orig.out_channels, orig.kernel_size, orig.stride, orig.padding, bias=False)
    elif 'efficientnet' in arch_name:
        model = torchvision.models.efficientnet_v2_s(weights=None, num_classes=num_classes)
        if input_channels != 3:
            orig = model.features[0][0]
            model.features[0][0] = nn.Conv2d(input_channels, orig.out_channels, orig.kernel_size, orig.stride, orig.padding, bias=False)
    elif 'googlenet' in arch_name:
        model = torchvision.models.googlenet(weights=None, aux_logits=False, num_classes=num_classes)
        if input_channels != 3:
            orig = model.conv1.conv
            model.conv1.conv = nn.Conv2d(input_channels, orig.out_channels, orig.kernel_size, orig.stride, orig.padding, bias=False)
    elif 'vgg' in arch_name:
        model = torchvision.models.vgg16_bn(weights=None, num_classes=num_classes)
        if input_channels != 3:
            orig = model.features[0]
            model.features[0] = nn.Conv2d(input_channels, orig.out_channels, orig.kernel_size, orig.stride, orig.padding, bias=False)
    else:
        model = torchvision.models.resnet18(weights=None, num_classes=num_classes)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, class_names, arch_name

In [ ]:
def parse_original_filename(filename):
    """
    Extracts the base filename.
    """
    if "_star_" in filename:
        return filename.split("_star_")[0]
    return filename

In [ ]:
# --- Multithreaded Data Loading ---

def load_single_image(file_path):
    """Worker function to decode XISF to numpy array."""
    try:
        # Use string path for xisf library compatibility
        f = xisf.XISF(str(file_path))
        img_data = f.read_image(0).data
        img_data = np.asarray(img_data, dtype=np.float32)
        
        # Handle dimensions (H, W)
        if img_data.ndim == 3:
            img_data = img_data.squeeze()
            
        # Add channel dim for PyTorch (H, W, C) for ToTensor
        # Because ToTensor expects (H, W, C) or (H, W)
        # We ensure it's (H, W, 1) to be explicit for grayscale
        img_data = img_data[..., np.newaxis] 
        
        return {
            "path": file_path,
            "data": img_data,
            "original_filename": parse_original_filename(file_path.name),
            "ground_truth": file_path.parent.name
        }
    except Exception as e:
        return None

def load_dataset_to_ram(image_files):
    dataset = []
    print(f"Loading {len(image_files)} images into RAM using {os.cpu_count()} workers...")
    
    with concurrent.futures.ProcessPoolExecutor() as executor:
        # Submit all jobs
        futures = [executor.submit(load_single_image, p) for p in image_files]
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(image_files), desc="Pre-loading RAM"):
            result = future.result()
            if result is not None:
                dataset.append(result)
                
    print(f"Successfully loaded {len(dataset)} images.")
    return dataset

# Dataset Wrapper
class InMemoryDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item["data"]
        if self.transform:
            image = self.transform(image)
        
        # Return metadata needed for tracking
        return {
            "tensor": image,
            "original_filename": item["original_filename"],
            "ground_truth": item["ground_truth"],
            "filename": item["path"].name
        }

In [ ]:
# 1. Find all Images
image_files = list(TRAINING_DATA_DIR.rglob(f"*{IMAGE_EXT}"))

# 2. Load to RAM
ram_dataset_list = load_dataset_to_ram(image_files)

# 3. Create Dataset & Loader
dataset = InMemoryDataset(ram_dataset_list, transform=transform)
data_loader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)

In [ ]:
# 4. Evaluation Loop (Batched)
model_files = list(TRAINING_DATA_DIR.glob("*.pth"))
final_results = []

print(f"Evaluating {len(model_files)} models on GPU batch size {BATCH_SIZE}...")

for model_path in tqdm(model_files, desc="Models"):
    try:
        model, class_names, _ = load_inference_model(model_path)
        model_name = model_path.name
        
        # Batched Inference
        with torch.no_grad():
            for batch in tqdm(data_loader, desc=f"Inference {model_name}", leave=False):
                images = batch["tensor"].to(device)
                original_filenames = batch["original_filename"]
                ground_truths = batch["ground_truth"]
                filenames = batch["filename"]
                
                outputs = model(images)
                _, predictions = torch.max(outputs, 1)
                
                # Collect batch results
                for i in range(len(predictions)):
                    pred_idx = predictions[i].item()
                    final_results.append({
                        "Model": model_name,
                        "OriginalFilename": original_filenames[i],
                        "ImageFile": filenames[i],
                        "GroundTruth": ground_truths[i],
                        "PredictedClass": class_names[pred_idx]
                    })

    except Exception as e:
        print(f"Failed model {model_path.name}: {e}")

In [17]:
# 5. Aggregation and Pivoting
df_raw = pd.DataFrame(final_results)

print("Aggregating consensus results...")

# Step A: Calculate Consensus per (Model Name, Original Filename)
# This collects votes from all star cutouts for a single image, for a single model.
consensus_rows = []
grouped = df_raw.groupby(["Model", "OriginalFilename"])

for (model_name, original_filename), group in grouped:
    ground_truth = group.iloc[0]["GroundTruth"]
    votes = group["PredictedClass"].tolist()
    vote_counts = Counter(votes)
    
    winner, count = vote_counts.most_common(1)[0]
    confidence = count / len(votes)
    
    consensus_rows.append({
        "Model": model_name,
        "OriginalFilename": original_filename,
        "GroundTruth": ground_truth,
        "Winner": winner,
        "Confidence": confidence,
        "IsMatch": winner == ground_truth
    })

df_consensus = pd.DataFrame(consensus_rows)

# Step B: Pivot to Final Report Structure
print("Pivoting report...")
pivot_data = []
unique_files = df_consensus["OriginalFilename"].unique()

for filename in tqdm(unique_files, desc="Building Pivot"):
    # Get all model results for this file
    file_data = df_consensus[df_consensus["OriginalFilename"] == filename]
    if file_data.empty: continue
        
    ground_truth = file_data.iloc[0]["GroundTruth"]
    
    # Calculate Overall Performance across models
    total_models = len(file_data)
    matches = file_data["IsMatch"].sum()
    overall_accuracy = matches / total_models if total_models > 0 else 0.0
    
    row = {
        "Base Filename": filename,
        "Base Class": ground_truth,
        "Overall Performance": overall_accuracy # Will format as % later
    }
    
    # Add Model Columns
    for _, row_data in file_data.iterrows():
        m_name = row_data["Model"]
        # Column Naming Scheme: ModelName [%, Class, Match]
        row[f"{m_name} %"] = row_data["Confidence"]
        row[f"{m_name} Class"] = row_data["Winner"]
        row[f"{m_name} Match"] = "MATCH" if row_data["IsMatch"] else "MISMATCH"
        
    pivot_data.append(row)

report_df = pd.DataFrame(pivot_data)

# Format percentages for display (optional, but good for CSV/Console)
# For Excel, we might want to keep them as numbers, but string is safer for quick viewing
# Let's keep them as floats for sorting, but maybe round them.
# Actually user asked for formulas, better to have numbers.

# Reorder Columns to put Base info first
base_cols = ["Base Filename", "Base Class", "Overall Performance"]
model_cols = sorted([c for c in report_df.columns if c not in base_cols])
report_df = report_df[base_cols + model_cols]

# Display Outliers (Low Overall Performance)
# Define outlier as < 50% agreement or something similar check
outliers = report_df[report_df["Overall Performance"] < 1.0]
print(f"Total Images: {len(report_df)}")
print(f"Images with any mismatch: {len(outliers)}")

print("\n--- Top Disagreements (Outliers) ---")
display(outliers.sort_values("Overall Performance").head())

Aggregating consensus results...
Pivoting report...


Building Pivot:   0%|          | 0/800 [00:00<?, ?it/s]

Total Images: 800
Images with any mismatch: 167

--- Top Disagreements (Outliers) ---


,Base Filename,Base Class,Overall Performance,Pre-Trained_MobileNetV2.pth %,Pre-Trained_MobileNetV2.pth Class,Pre-Trained_MobileNetV2.pth Match,Pre-Trained_Resnet-18.pth %,Pre-Trained_Resnet-18.pth Class,Pre-Trained_Resnet-18.pth Match,Scratch_DenseNet-121.pth %,...,Scratch_MobileNetV2.pth Match,Scratch_ResNet-18.pth %,Scratch_ResNet-18.pth Class,Scratch_ResNet-18.pth Match,Scratch_ShuffleNetV2.pth %,Scratch_ShuffleNetV2.pth Class,Scratch_ShuffleNetV2.pth Match,Scratch_VGG16_BN.pth %,Scratch_VGG16_BN.pth Class,Scratch_VGG16_BN.pth Match
291,2024-10-14_05-10-44__-0.00_30.00s_0123_debayered,focus,0.0,0.800000,good,MISMATCH,0.933333,good,MISMATCH,0.966667,...,MISMATCH,0.900000,good,MISMATCH,0.833333,good,MISMATCH,1.000000,good,MISMATCH
727,2025-06-10_02-00-47_North America Nebula_0.00_...,wind,0.0,1.000000,good,MISMATCH,0.966667,good,MISMATCH,1.000000,...,MISMATCH,1.000000,good,MISMATCH,1.000000,good,MISMATCH,0.966667,good,MISMATCH
791,2025-06-11_03-26-47_North America Nebula_0.00_...,tracking,0.0,1.000000,good,MISMATCH,1.000000,good,MISMATCH,1.000000,...,MISMATCH,1.000000,good,MISMATCH,1.000000,good,MISMATCH,1.000000,good,MISMATCH
738,2025-06-10_03-15-55_North America Nebula_-0.00...,wind,0.0,0.966667,good,MISMATCH,1.000000,good,MISMATCH,1.000000,...,MISMATCH,0.933333,good,MISMATCH,0.933333,good,MISMATCH,0.800000,good,MISMATCH
562,2024-10-21_03-24-21_Great Orion Nebula_0.00_30...,focus,0.1,0.566667,good,MISMATCH,0.533333,good,MISMATCH,0.500000,...,MISMATCH,0.500000,wind,MISMATCH,0.533333,wind,MISMATCH,0.433333,good,MISMATCH


In [18]:
# 6. Save to Excel with formatting
output_xlsx = TRAINING_DATA_DIR / "training_data_eval_report.xlsx"

try:
    # We can use Pandas Styler if displaying, but for saving just raw values is often best
    # so user can apply their own conditional formatting
    report_df.to_excel(output_xlsx, index=False)
    print(f"\nSaved Pivot Report to: {output_xlsx}")
except ImportError:
    print("openpyxl missing. Saving as CSV.")
    report_df.to_csv(str(output_xlsx).replace('.xlsx', '.csv'), index=False)


Saved Pivot Report to: /Storage/Files/practicalML/gitlab/practicalml/training_data/training_data_eval_report.xlsx
